# P03 Fixed-Sample Acceptance Baseline v2

## tl;dr

Ten local PDF fixtures are measured at atomic Fact and logical Information Point grain. The frozen baseline records source mix, review volume, missing fields and conservative P01 entity attribution. The 20–50 range is a coverage target, never a quota.

## Context & Methods

The notebook runs `scripts/verify_p03_facts.py` against `docs/beta02-acceptance/samples.json` and its local PDFs. Each fixture hash is verified before parsing. Project and Asset Unit matching uses the fixed nine-Project/five-Asset P01 catalog.

### Key assumptions

- Review count is measured at logical Information Point grain.
- Summary includes Summary, Highlights, Key Results and Executive Summary.
- Table means a recognized standard ASX form.
- Zero points can be correct for routing-only negative samples such as Appendix 3Y.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd

root = Path.cwd().resolve()
if not (root / 'scripts' / 'verify_p03_facts.py').exists():
    root = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(root / 'scripts'))
from verify_p03_facts import build_current_baseline
from app.services.p3_acceptance_baseline import compare_fixed_sample_baseline

expected_path = root / 'docs/beta02-acceptance/p03-fixed-sample-baseline-v2.json'
expected = json.loads(expected_path.read_text(encoding='utf-8'))
actual = build_current_baseline()
drift = compare_fixed_sample_baseline(actual, expected)
print('Source:', expected_path)
print('Samples:', actual['sample_count'])
print('Baseline drift fields:', len(drift))

## Data

The bounded table below contains the requested acceptance measurements for every fixed sample.

In [ ]:
rows = []
for sample in actual['samples']:
    projects = [item['project'] for item in sample['attribution']['projects']]
    assets = [item['asset_unit'] for item in sample['attribution']['asset_units']]
    missing = sample['unfound_fields']['family_rule_slots'] + sample['unfound_fields']['required_fact_types']
    rows.append({
        'Sample': sample['sample_id'], 'Facts': sample['atomic_fact_count'],
        'Points': sample['information_point_count'],
        'Summary': sample['summary_information_point_count'],
        'Table': sample['table_information_point_count'],
        'Body': sample['body_fallback_information_point_count'],
        'Review': sample['review_needed_count'],
        'Missing': ', '.join(missing) or '-',
        'Projects': ', '.join(projects) or 'Company',
        'Assets': ', '.join(assets) or '-',
    })
table = pd.DataFrame(rows)
print(table.to_string(index=False))

## Results

In [ ]:
assert drift == []
assert all(sample['fixture_sha256_verified'] for sample in actual['samples'])
assert all(sample['reconciliation']['all_atomic_facts_grouped'] for sample in actual['samples'])
assert all(sample['reconciliation']['all_points_source_counted'] for sample in actual['samples'])
assert actual['counting_policy']['target_is_not_a_quota'] is True
print('All acceptance-baseline assertions passed.')

## Takeaways

- S05 preserves 74 atomic Facts but presents 18 logical drilling information points.
- S02 correctly attributes 14 points to the Appendix 5B table rather than Summary.
- S08 correctly produces no business Facts as an Appendix 3Y negative-routing sample.
- S10 exposes the current high-priority gap: Permit Type and Permit Status are not yet extracted.
- Unconfirmed attribution remains at Company level; no acceptance run creates a Project or Asset Unit.